# Notebook 09 v5 — Frozen Abstention-Policy Post-hoc Re-evaluation

This notebook applies the unchanged frozen Stage 07 XGBoost `I_full_40`
pipeline and the exact frozen Stage 08 abstention policy to the previously
inspected 2021–2024 period.

The frozen post-model decision policy is:

```text
Recovery-only Breadth gate
AND fixed raw-score cutoff
AND daily Top-5% maximum cap
minimum signals per date = 0
```

## Scientific status

This calendar period is **not statistically untouched**. Earlier project
iterations and the Breadth design process inspected results from the same
2021–2024 period. Therefore:

- this run is post-hoc and diagnostic;
- confirmatory or external-validation claims are not allowed;
- Stage 09 performs no policy search, threshold search, calibration, or model
  retraining;
- genuine confirmation requires later unused data or prospective paper
  trading.

Scores and policy decisions are written to an outcome-free lock before test
labels and corrected event outcomes are opened.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import joblib
import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("joblib:", joblib.__version__)


Python: 3.12.2
numpy: 1.26.4
pandas: 2.2.3
joblib: 1.4.2


## 1. Locate repository and import frozen project modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.probability_policy import probability_metrics
from src.evaluation.signal_policy import binary_signal_metrics
from src.evaluation.unseen_test_signal import (
    file_sha256,
    corrected_event_outcome_metrics,
    grouped_corrected_event_outcome_metrics,
    reconstruct_corrected_event_outcomes,
)
from src.features.leakage_checks import (
    ConfirmedZigZagConfig,
    build_candidate_long_mask,
    build_confirmation_gated_zigzag_state,
)
from src.features.preprocessing import (
    CARRIED_STAGE04_NUMERIC_FEATURES,
    EMA_WINDOWS,
    ENGINEERED_MODEL_FEATURES,
    FINAL_CATEGORICAL_FEATURES as BASE_CATEGORICAL_FEATURES,
    FINAL_MODEL_FEATURES as BASE_MODEL_FEATURES,
    FINAL_NUMERIC_FEATURES as BASE_NUMERIC_FEATURES,
    MARKET_INDEX_REQUIRED_COLUMNS,
    RAW_REQUIRED_COLUMNS,
    FeatureEngineeringConfig,
    build_canonical_market_index,
    build_final_feature_frame,
    build_market_regime_feature_frame,
    parse_market_date,
    prepare_market_index_source,
)
from src.features.stage04_breadth_extension import (
    STAGE04_BREADTH_CATEGORICAL_FEATURES,
    STAGE04_BREADTH_FEATURES,
    STAGE04_BREADTH_NUMERIC_FEATURES,
    Stage04BreadthConfig,
    build_daily_market_breadth,
)
from src.features.unseen_breadth import (
    UNSEEN_BREADTH_SCHEMA_VERSION,
    load_started_run_length,
    merge_stage04_breadth_features,
    prepare_symbol_breadth_observations,
)
from src.models.abstention_policy import (
    ABSTENTION_POLICY_SCHEMA_VERSION,
    AbstentionPolicy,
    apply_abstention_policy_inference,
)
from src.models.frozen_training import dataframe_fingerprint
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen model lineage and exact Stage 08 abstention policy


In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = json.load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"JSON artifact must be an object: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")
stage09_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "unseen_test_evaluation.yaml"
)

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"
LABELED_TEST_PATH = DATA_PATHS["labeled_data"] / "unseen_test"
RAW_DATA_PATH = DATA_PATHS["raw_data"]

frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
stage04_policy_path = RESULT_PATHS["manifests"] / "04_candidate_filter_policy.json"
stage07_manifest_path = RESULT_PATHS["manifests"] / "07_frozen_model_training_manifest.json"
stage08_policy_path = RESULT_PATHS["manifests"] / "08_abstention_signal_policy.json"
stage08_manifest_path = RESULT_PATHS["manifests"] / "08_abstention_policy_manifest.json"
model_path = RESULT_PATHS["models"] / "07_frozen_xgboost_pipeline.joblib"

for path in [
    frozen_universe_path,
    stage04_policy_path,
    stage07_manifest_path,
    stage08_policy_path,
    stage08_manifest_path,
    model_path,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required frozen input is missing: {path}")

frozen_universe_df = pd.read_csv(frozen_universe_path, low_memory=False)
stage04_policy = load_json(stage04_policy_path)
stage07_manifest = load_json(stage07_manifest_path)
stage08_policy = load_json(stage08_policy_path)
stage08_manifest = load_json(stage08_manifest_path)

TEST_START = pd.Timestamp(stage09_config["temporal_scope"]["unseen_test_start"])
SIGNAL_GENERATION_END = pd.Timestamp(
    stage09_config["temporal_scope"]["signal_generation_end"]
)
OUTCOME_OBSERVATION_TAIL_END = pd.Timestamp(
    stage09_config["temporal_scope"]["outcome_observation_tail_end"]
)
TEST_END = SIGNAL_GENERATION_END
TRAIN_END = pd.Timestamp(stage09_config["temporal_scope"]["train_end"])

frozen_symbols = sorted(frozen_universe_df["symbol"].astype(str).tolist())
frozen_symbol_set = set(frozen_symbols)

SELECTED_MODEL = str(stage08_policy["source_model"])
SELECTED_FEATURE_SET = str(stage08_policy["source_feature_set"])
SELECTED_TRIAL = int(stage08_policy["source_trial"])
SELECTED_FEATURES = list(stage07_manifest["features"]["model_features"])
SELECTED_NUMERIC_FEATURES = list(
    stage07_manifest["features"]["numeric_features"]
)
SELECTED_CATEGORICAL_FEATURES = list(
    stage07_manifest["features"]["categorical_features"]
)

STAGE08_POLICY_ID = str(stage08_manifest["selected_policy_id"])
STAGE08_POLICY_CONFIGURATION_HASH = str(
    stage08_policy["configuration_hash"]
)
STAGE08_POLICY_FILE_SHA256 = file_sha256(stage08_policy_path)

POLICY_GATE_NAME = str(stage08_policy["market_gate"]["gate_name"])
POLICY_ALLOWED_REGIMES = tuple(
    str(value)
    for value in stage08_policy["market_gate"]["allowed_regimes"]
)
POLICY_MINIMUM_RAW_SCORE = float(
    stage08_policy["score_threshold"]["minimum_raw_score"]
)
POLICY_THRESHOLD_SOURCE_QUANTILE = float(
    stage08_policy["score_threshold"][
        "source_quantile_of_baseline_selected_oof_scores"
    ]
)
POLICY_MAXIMUM_DAILY_FRACTION = float(
    stage08_policy["daily_cap"]["maximum_fraction"]
)
POLICY_MINIMUM_SIGNALS_PER_DATE = int(
    stage08_policy["daily_cap"]["minimum_signals_per_date"]
)
POLICY_RANKING_ORDER = list(
    stage08_policy["daily_cap"]["ranking_order"]
)

FROZEN_ABSTENTION_POLICY = AbstentionPolicy(
    gate_name=POLICY_GATE_NAME,
    allowed_regimes=POLICY_ALLOWED_REGIMES,
    minimum_score=POLICY_MINIMUM_RAW_SCORE,
    maximum_daily_fraction=POLICY_MAXIMUM_DAILY_FRACTION,
    minimum_signals_per_date=POLICY_MINIMUM_SIGNALS_PER_DATE,
    score_column="xgboost_ranking_score",
)

assert stage09_config["status"] == (
    "stage_09_configured_v5_abstention_policy_post_hoc_retest"
)
assert stage09_config["scientific_status"][
    "prior_test_period_previously_inspected"
] is True
assert stage09_config["scientific_status"][
    "abstention_policy_selected_only_from_train_oof"
] is True
assert stage09_config["scientific_status"][
    "confirmatory_claim_allowed"
] is False
assert UNSEEN_BREADTH_SCHEMA_VERSION == (
    "stage09_v4_causal_stage04_breadth_reconstruction"
)
assert ABSTENTION_POLICY_SCHEMA_VERSION == (
    "stage08_v3_train_only_breadth_gate_threshold_daily_cap"
)
assert OUTCOME_OBSERVATION_TAIL_END >= SIGNAL_GENERATION_END
assert len(frozen_symbols) == int(
    stage09_config["frozen_inputs"]["expected_universe_size"]
)

assert stage04_policy["status"] == "frozen_after_integrity_checks"
assert stage04_policy["primary_side"] == "long"
assert np.isclose(float(stage04_policy["primary_threshold_fraction"]), 0.15)

assert stage07_manifest["status"] == "frozen_after_integrity_checks"
assert stage07_manifest["model"]["selected_model"] == "xgboost"
assert stage07_manifest["model"]["selected_feature_set"] == "I_full_40"
assert int(stage07_manifest["model"]["selected_trial_number"]) == SELECTED_TRIAL
assert stage07_manifest["unseen_test_used"] is False

assert stage08_policy["status"] == "frozen_train_only_abstention_policy"
assert stage08_manifest["status"] == "frozen_after_integrity_checks"
assert stage08_manifest["selected_policy_id"] == STAGE08_POLICY_ID
assert stage08_manifest[
    "selected_policy_configuration_hash"
] == STAGE08_POLICY_CONFIGURATION_HASH
assert stage08_manifest["selected_policy"][
    "configuration_hash"
] == STAGE08_POLICY_CONFIGURATION_HASH
assert stage08_policy["source_model_sha256"] == stage07_manifest[
    "model"
]["model_sha256"]
assert stage08_policy["score_policy"]["interpretation"] == (
    "uncalibrated_ranking_score"
)
assert stage08_policy["score_policy"]["calibrator_fitted"] is False
assert stage08_policy["score_policy"][
    "literal_probability_interpretation_allowed"
] is False
assert stage08_policy["score_policy"][
    "threshold_is_operational_cutoff_not_calibrated_probability"
] is True
assert stage08_policy["daily_cap"]["zero_signal_dates_allowed"] is True
assert stage08_policy["safeguards"]["unseen_test_loaded"] is False
assert stage08_policy["safeguards"]["model_retrained"] is False

expected_policy = stage09_config["frozen_signal_policy"]
assert STAGE08_POLICY_ID == expected_policy["selected_policy_id"]
assert POLICY_GATE_NAME == expected_policy["expected_gate_name"]
assert list(POLICY_ALLOWED_REGIMES) == list(
    expected_policy["expected_allowed_regimes"]
)
assert np.isclose(
    POLICY_MINIMUM_RAW_SCORE,
    float(expected_policy["expected_minimum_raw_score"]),
)
assert np.isclose(
    POLICY_THRESHOLD_SOURCE_QUANTILE,
    float(expected_policy["expected_threshold_source_quantile"]),
)
assert np.isclose(
    POLICY_MAXIMUM_DAILY_FRACTION,
    float(expected_policy["expected_maximum_daily_fraction"]),
)
assert POLICY_MINIMUM_SIGNALS_PER_DATE == int(
    expected_policy["expected_minimum_signals_per_date"]
)
assert POLICY_MINIMUM_SIGNALS_PER_DATE == 0
assert POLICY_RANKING_ORDER == list(expected_policy["ranking_order"])

assert SELECTED_MODEL == "xgboost"
assert SELECTED_FEATURE_SET == "I_full_40"
assert SELECTED_TRIAL == 10
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert list(stage07_manifest["features"]["model_features"]) == SELECTED_FEATURES
assert list(stage07_manifest["features"]["numeric_features"]) == (
    SELECTED_NUMERIC_FEATURES
)
assert list(stage07_manifest["features"]["categorical_features"]) == (
    SELECTED_CATEGORICAL_FEATURES
)

expected_model_hash = str(
    stage09_config["frozen_inputs"]["expected_model_sha256"]
)
actual_model_hash = file_sha256(model_path)
assert actual_model_hash == expected_model_hash
assert actual_model_hash == stage07_manifest["model"]["model_sha256"]

lineage_audit_df = pd.DataFrame([{
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_claim_allowed": False,
    "stage04_candidate_threshold_fraction": float(
        stage04_policy["primary_threshold_fraction"]
    ),
    "stage07_code_sha": stage07_manifest["git_commit_sha"],
    "stage07_model_sha256": actual_model_hash,
    "stage07_selected_model": SELECTED_MODEL,
    "stage07_selected_feature_set": SELECTED_FEATURE_SET,
    "stage07_selected_trial": SELECTED_TRIAL,
    "stage07_raw_features": len(SELECTED_FEATURES),
    "stage08_code_sha": stage08_manifest["git_commit_sha"],
    "stage08_policy_id": STAGE08_POLICY_ID,
    "stage08_policy_file_sha256": STAGE08_POLICY_FILE_SHA256,
    "stage08_policy_configuration_hash": STAGE08_POLICY_CONFIGURATION_HASH,
    "stage08_gate_name": POLICY_GATE_NAME,
    "stage08_allowed_regimes": json.dumps(
        list(POLICY_ALLOWED_REGIMES),
        ensure_ascii=False,
    ),
    "stage08_minimum_raw_score": POLICY_MINIMUM_RAW_SCORE,
    "stage08_threshold_source_quantile": POLICY_THRESHOLD_SOURCE_QUANTILE,
    "stage08_maximum_daily_fraction": POLICY_MAXIMUM_DAILY_FRACTION,
    "stage08_minimum_signals_per_date": POLICY_MINIMUM_SIGNALS_PER_DATE,
    "fixed_raw_score_threshold_selected": True,
    "fixed_probability_threshold_selected": False,
    "policy_reselected_in_stage09": False,
    "model_retrained": False,
    "test_outcomes_used_before_score_lock": False,
}])
lineage_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_frozen_input_lineage_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(lineage_audit_df)


,scientific_status,confirmatory_claim_allowed,stage04_candidate_threshold_fraction,stage07_code_sha,stage07_model_sha256,stage07_selected_model,stage07_selected_feature_set,stage07_selected_trial,stage07_raw_features,stage08_code_sha,...,stage08_allowed_regimes,stage08_minimum_raw_score,stage08_threshold_source_quantile,stage08_maximum_daily_fraction,stage08_minimum_signals_per_date,fixed_raw_score_threshold_selected,fixed_probability_threshold_selected,policy_reselected_in_stage09,model_retrained,test_outcomes_used_before_score_lock
0,post_hoc_retest_previously_inspected_period,False,0.15,4a6b344fc6219e08b32586002922f390061d285e,73a781551cdabd6bb67f5b9c0836a8683372e46487d9aa...,xgboost,I_full_40,10,40,964ca042fa030830fab955f50a477ee69c4790f2,...,"[""recovery_negative"", ""recovery_positive""]",0.384463,0.0,0.05,0,True,False,False,False,False


## 3. Validate train/test/raw file inventories

In [4]:
def symbol_from_path(path: Path, suffix: str) -> str:
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected file name: {path.name}")
    return path.name[:-len(suffix)]


train_files = sorted(LABELED_TRAIN_PATH.glob("*_train_labeled.csv"))
test_files = sorted(LABELED_TEST_PATH.glob("*_unseen_test_labeled.csv"))
raw_files = sorted(RAW_DATA_PATH.glob("*.csv"))

train_file_map = {
    symbol_from_path(path, "_train_labeled.csv"): path
    for path in train_files
}
test_file_map = {
    symbol_from_path(path, "_unseen_test_labeled.csv"): path
    for path in test_files
}
raw_file_map = {path.stem: path for path in raw_files}

assert set(train_file_map) == frozen_symbol_set
assert set(test_file_map) == frozen_symbol_set
assert set(raw_file_map).issuperset(frozen_symbol_set)

inventory_audit_df = pd.DataFrame([{
    "frozen_symbols": len(frozen_symbols),
    "labeled_train_files": len(train_file_map),
    "labeled_unseen_test_files": len(test_file_map),
    "raw_files_covering_frozen_universe": int(len(frozen_symbol_set.intersection(raw_file_map))),
    "test_start": TEST_START,
    "signal_generation_end": SIGNAL_GENERATION_END,
    "outcome_observation_tail_end": OUTCOME_OBSERVATION_TAIL_END,
}])
inventory_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_inventory_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(inventory_audit_df)


,frozen_symbols,labeled_train_files,labeled_unseen_test_files,raw_files_covering_frozen_universe,test_start,signal_generation_end,outcome_observation_tail_end
0,499,499,499,499,2021-03-21,2024-09-22,2024-10-26


## 4. Build causal market-index and Stage 04 Breadth history through the signal end

Both market feature families use raw observations no later than
22 September 2024. Breadth is reconstructed from previous-valid adjusted-close
returns over the frozen 499-symbol universe using the exact Stage 04 formula.


In [5]:
feature_config = FeatureEngineeringConfig(
    relative_strength_window=12,
    return_zscore_window=12,
    volume_window=30,
    market_volatility_window=20,
    market_ema_fast_window=20,
    market_ema_slow_window=60,
    market_drawdown_window=60,
    market_index_consistency_relative_tolerance=1.0e-10,
)
breadth_config = Stage04BreadthConfig()

market_index_parts = []
breadth_observation_parts = []
market_index_error_rows = []
market_rows_after_test_end = 0
started = time.perf_counter()

for file_number, symbol in enumerate(frozen_symbols, start=1):
    raw_path = raw_file_map[symbol]
    try:
        raw_market = pd.read_csv(
            raw_path,
            usecols=list(MARKET_INDEX_REQUIRED_COLUMNS),
            low_memory=False,
        )
        raw_dates = parse_market_date(raw_market["dEven"])
        market_rows_after_test_end += int(raw_dates.gt(TEST_END).sum())
        raw_market = raw_market.loc[
            raw_dates.notna() & raw_dates.le(TEST_END)
        ].copy()
        market_index_parts.append(
            prepare_market_index_source(raw_market, source_symbol=symbol)
        )
        breadth_observation_parts.append(
            prepare_symbol_breadth_observations(
                raw_path,
                symbol=symbol,
                horizon_end=TEST_END,
                config=breadth_config,
            )
        )
    except Exception as exc:
        market_index_error_rows.append({
            "symbol": symbol,
            "file_name": raw_path.name,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

    if file_number == 1 or file_number % 50 == 0 or file_number == len(frozen_symbols):
        print(
            f"[market history] [{file_number:>4}/{len(frozen_symbols)}] "
            f"elapsed={time.perf_counter() - started:,.1f}s "
            f"errors={len(market_index_error_rows)}"
        )

market_index_errors_df = pd.DataFrame(
    market_index_error_rows,
    columns=["symbol", "file_name", "error_type", "error_message"],
)
market_index_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_market_index_source_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not market_index_errors_df.empty:
    raise RuntimeError("Market-index or Breadth source errors exist.")

market_observation_panel = pd.concat(market_index_parts, ignore_index=True)
canonical_market_index_df, market_consistency_audit_df = build_canonical_market_index(
    market_observation_panel,
    relative_tolerance=feature_config.market_index_consistency_relative_tolerance,
)
market_regime_feature_frame, market_regime_audit = build_market_regime_feature_frame(
    canonical_market_index_df,
    config=feature_config,
)

breadth_observation_panel = pd.concat(
    breadth_observation_parts,
    ignore_index=True,
)
market_breadth_feature_frame, breadth_audit = build_daily_market_breadth(
    breadth_observation_panel,
    config=breadth_config,
)

market_consistency_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_market_index_consistency_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
market_regime_audit_df = pd.DataFrame([{
    **market_regime_audit,
    "test_feature_horizon_end": TEST_END,
    "raw_rows_excluded_after_test_end": market_rows_after_test_end,
    "inconsistent_cross_file_rows": int(
        market_consistency_audit_df["inconsistent_across_raw_files"].sum()
    ),
}])
market_regime_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_market_regime_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

breadth_audit_df = pd.DataFrame([{
    **breadth_audit,
    "schema_version": UNSEEN_BREADTH_SCHEMA_VERSION,
    "frozen_symbols": len(frozen_symbols),
    "feature_horizon_end": TEST_END,
    "same_day_or_past_only": True,
    "future_rows_used": False,
}])
breadth_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_market_breadth_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert canonical_market_index_df["dEven"].max() <= TEST_END
assert market_regime_feature_frame["dEven"].max() <= TEST_END
assert market_regime_feature_frame["dEven"].is_unique
assert market_regime_feature_frame["dEven"].is_monotonic_increasing

assert market_breadth_feature_frame["dEven"].max() <= TEST_END
assert market_breadth_feature_frame["dEven"].is_unique
assert market_breadth_feature_frame["dEven"].is_monotonic_increasing
assert market_breadth_feature_frame["market_breadth_raw"].between(
    -1.0, 1.0, inclusive="both"
).all()
assert market_breadth_feature_frame[
    [
        "market_breadth_raw",
        "market_breadth_ema30",
        "market_breadth_slope5",
        "market_breadth_regime",
    ]
].notna().all().all()

display(market_regime_audit_df)
display(breadth_audit_df)


[market history] [   1/499] elapsed=0.1s errors=0
[market history] [  50/499] elapsed=5.3s errors=0
[market history] [ 100/499] elapsed=11.3s errors=0
[market history] [ 150/499] elapsed=17.5s errors=0
[market history] [ 200/499] elapsed=23.2s errors=0
[market history] [ 250/499] elapsed=28.8s errors=0
[market history] [ 300/499] elapsed=34.7s errors=0
[market history] [ 350/499] elapsed=40.1s errors=0
[market history] [ 400/499] elapsed=45.9s errors=0
[market history] [ 450/499] elapsed=51.4s errors=0
[market history] [ 499/499] elapsed=56.6s errors=0


,market_calendar_rows,market_first_date,market_last_date,invalid_market_ohlc_rows,locked_market_rows,locked_market_up_rows,locked_market_down_rows,locked_market_equal_previous_rows,market_close_location_missing_rows,market_return_1_missing_rows,market_return_5_missing_rows,market_return_20_missing_rows,market_volatility_20_missing_rows,market_ema_20_distance_missing_rows,market_ema_60_distance_missing_rows,market_range_fraction_missing_rows,market_drawdown_60_missing_rows,test_feature_horizon_end,raw_rows_excluded_after_test_end,inconsistent_cross_file_rows
0,2520,2014-03-19,2024-09-22,8,228,113,114,0,9,1,5,20,20,19,59,8,59,2024-09-22,165780,0


,calendar_rows,first_date,last_date,minimum_valid_symbols,maximum_valid_symbols,raw_missing_rows,ema30_missing_rows_before_warmup_encoding,slope5_missing_rows_before_warmup_encoding,warmup_regime_rows,warmup_numeric_fill_value,schema_version,frozen_symbols,feature_horizon_end,same_day_or_past_only,future_rows_used
0,2519,2014-03-25,2024-09-22,6,490,0,29,34,34,0.0,stage09_v4_causal_stage04_breadth_reconstruction,499,2024-09-22,True,False


## 5. Build causal 40-feature candidate rows without loading outcomes

The train history is prepended as rolling warm-up. The original `started`
run length is loaded without recomputation. The five Stage 04 additions are
merged after the unchanged +15% ZigZag candidate rule is applied.


In [6]:
zigzag_policy = columns_config["zigzag_audit"]
zigzag_config = ConfirmedZigZagConfig(
    depth=int(zigzag_policy["source_depth"]),
    deviation_percent=float(zigzag_policy["source_deviation_percent"]),
)
PRIMARY_THRESHOLD = float(stage04_policy["primary_threshold_fraction"])

history_feature_columns = [
    "dEven", "adj_high", "adj_low", "adj_last_price", "RSI_14", "macd",
    *[f"EMA_{window}" for window in EMA_WINDOWS],
]
test_inference_columns = [*history_feature_columns, "eligible_for_modeling"]

candidate_parts = []
feature_audit_rows = []
candidate_audit_rows = []
candidate_error_rows = []
started_source_counts = {}
total_test_rows = 0
total_test_eligible = 0
started = time.perf_counter()

for file_number, symbol in enumerate(frozen_symbols, start=1):
    train_path = train_file_map[symbol]
    test_path = test_file_map[symbol]
    raw_path = raw_file_map[symbol]
    try:
        train_history = pd.read_csv(
            train_path,
            usecols=history_feature_columns,
            low_memory=False,
        )
        test_history = pd.read_csv(
            test_path,
            usecols=test_inference_columns,
            low_memory=False,
        )
        train_history["partition"] = "train"
        train_history["eligible_for_modeling"] = False
        test_history["partition"] = "unseen_test"

        full_history = pd.concat([train_history, test_history], ignore_index=True)
        full_history["dEven"] = pd.to_datetime(
            full_history["dEven"], errors="raise"
        ).dt.normalize()
        full_history = full_history.sort_values(
            ["dEven", "partition"], kind="stable"
        ).drop_duplicates(
            subset=["dEven"], keep="last"
        ).reset_index(drop=True)
        full_history = full_history.loc[
            full_history["dEven"].le(TEST_END)
        ].reset_index(drop=True)

        test_mask = full_history["partition"].eq("unseen_test")
        symbol_test_rows = int(test_mask.sum())
        symbol_test_eligible = int(
            full_history.loc[test_mask, "eligible_for_modeling"]
            .fillna(False).astype(bool).sum()
        )
        total_test_rows += symbol_test_rows
        total_test_eligible += symbol_test_eligible

        raw_frame = pd.read_csv(
            raw_path,
            usecols=list(RAW_REQUIRED_COLUMNS),
            low_memory=False,
        )
        raw_dates = parse_market_date(raw_frame["dEven"])
        raw_frame = raw_frame.loc[
            raw_dates.notna() & raw_dates.le(TEST_END)
        ].copy()

        final_feature_frame, feature_audit = build_final_feature_frame(
            full_history[history_feature_columns],
            raw_frame,
            market_regime_feature_frame,
            config=feature_config,
        )
        zigzag_state = build_confirmation_gated_zigzag_state(
            full_history[["dEven", "adj_high", "adj_low", "adj_last_price"]],
            config=zigzag_config,
            date_column="dEven",
        )

        eligible_series = full_history["eligible_for_modeling"].fillna(False).astype(bool)
        candidate_mask = build_candidate_long_mask(
            zigzag_state,
            eligible_for_modeling=eligible_series,
            threshold_fraction=PRIMARY_THRESHOLD,
        ) & test_mask

        candidate_dates = full_history.loc[candidate_mask, "dEven"].reset_index(drop=True)
        state_view = zigzag_state.loc[candidate_mask].reset_index(drop=True)
        feature_view = final_feature_frame.loc[
            final_feature_frame["dEven"].isin(candidate_dates)
        ].sort_values("dEven", kind="stable").reset_index(drop=True)

        candidate_frame = pd.DataFrame({
            "event_id": symbol + "::" + candidate_dates.dt.strftime("%Y-%m-%d"),
            "symbol": symbol,
            "dEven": candidate_dates,
        })
        candidate_frame[list(CARRIED_STAGE04_NUMERIC_FEATURES)] = (
            state_view[list(CARRIED_STAGE04_NUMERIC_FEATURES)]
        )
        candidate_frame = candidate_frame.merge(
            feature_view[["dEven", *ENGINEERED_MODEL_FEATURES]],
            on="dEven",
            how="left",
            validate="one_to_one",
        )

        started_frame, started_source_name = load_started_run_length(
            symbol=symbol,
            raw_path=raw_path,
            fallback_path=test_path,
            horizon_end=TEST_END,
            config=breadth_config,
        )
        started_source_counts[started_source_name] = (
            started_source_counts.get(started_source_name, 0) + 1
        )
        candidate_frame, breadth_merge_audit = merge_stage04_breadth_features(
            candidate_frame,
            started_frame=started_frame,
            breadth_frame=market_breadth_feature_frame,
            config=breadth_config,
        )

        missing_features = sorted(
            set(SELECTED_FEATURES) - set(candidate_frame.columns)
        )
        if missing_features:
            raise KeyError(
                f"Selected candidate features are missing: {missing_features}"
            )

        candidate_frame = candidate_frame[
            ["event_id", "symbol", "dEven", *SELECTED_FEATURES]
        ].copy()
        candidate_parts.append(candidate_frame)

        numeric_array = candidate_frame[
            SELECTED_NUMERIC_FEATURES
        ].to_numpy(dtype=float)

        feature_audit_rows.append({
            "symbol": symbol,
            "full_history_rows": int(len(full_history)),
            "test_rows": symbol_test_rows,
            "test_eligible_events": symbol_test_eligible,
            "candidate_events": int(len(candidate_frame)),
            "candidate_rows_with_any_feature_missing": int(
                candidate_frame[SELECTED_FEATURES].isna().any(axis=1).sum()
            ),
            "candidate_nonfinite_numeric_values": int(
                np.isinf(numeric_array).sum()
            ),
            "started_source": started_source_name,
            "breadth_identity_preserved": bool(
                breadth_merge_audit["candidate_identity_preserved"]
            ),
            **feature_audit,
        })
        candidate_audit_rows.append({
            "symbol": symbol,
            "test_rows": symbol_test_rows,
            "test_eligible_events": symbol_test_eligible,
            "candidate_events": int(len(candidate_frame)),
            "first_candidate_date": candidate_frame["dEven"].min() if len(candidate_frame) else pd.NaT,
            "last_candidate_date": candidate_frame["dEven"].max() if len(candidate_frame) else pd.NaT,
        })

    except Exception as exc:
        candidate_error_rows.append({
            "symbol": symbol,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

    if file_number == 1 or file_number % 25 == 0 or file_number == len(frozen_symbols):
        print(
            f"[test candidates] [{file_number:>4}/{len(frozen_symbols)}] "
            f"elapsed={time.perf_counter() - started:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=["symbol", "error_type", "error_message"],
)
candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_candidate_generation_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not candidate_errors_df.empty:
    display(candidate_errors_df.head(20))
    raise RuntimeError("Unseen-period candidate generation errors exist.")

candidate_panel = pd.concat(candidate_parts, ignore_index=True).sort_values(
    ["dEven", "symbol", "event_id"], kind="stable"
).reset_index(drop=True)
feature_engineering_audit_df = pd.DataFrame(feature_audit_rows)
candidate_by_symbol_audit_df = pd.DataFrame(candidate_audit_rows)
feature_engineering_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_feature_engineering_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
candidate_by_symbol_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_candidate_by_symbol_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert total_test_rows == int(stage09_config["frozen_inputs"]["expected_unseen_test_rows"])
assert total_test_eligible == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_eligible_events"]
)
assert len(candidate_panel) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert candidate_panel["event_id"].is_unique
assert candidate_panel["dEven"].between(TEST_START, TEST_END).all()
assert len(BASE_MODEL_FEATURES) == 35
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert set(STAGE04_BREADTH_FEATURES).issubset(SELECTED_FEATURES)

# Match the frozen Stage 07 preprocessing contract:
# - numeric missing values are handled by the already-fitted median imputer;
# - categorical missing values are handled by the already-fitted
#   most-frequent imputer;
# - infinities are never permitted.
#
# Therefore, do not drop candidate rows and do not fill values manually here.
for feature in SELECTED_NUMERIC_FEATURES:
    candidate_panel[feature] = pd.to_numeric(
        candidate_panel[feature],
        errors="coerce",
    )
for feature in SELECTED_CATEGORICAL_FEATURES:
    candidate_panel[feature] = (
        candidate_panel[feature].astype("string")
    )

missing_by_feature = (
    candidate_panel[SELECTED_FEATURES]
    .isna()
    .sum()
    .astype(int)
    .sort_values(ascending=False)
)
candidate_missingness_audit_df = pd.DataFrame({
    "feature": missing_by_feature.index,
    "missing_rows": missing_by_feature.to_numpy(dtype=int),
})
candidate_missingness_audit_df["candidate_rows"] = int(
    len(candidate_panel)
)
candidate_missingness_audit_df["missing_fraction"] = (
    candidate_missingness_audit_df["missing_rows"]
    / candidate_missingness_audit_df["candidate_rows"]
)
candidate_missingness_audit_df["feature_role"] = np.where(
    candidate_missingness_audit_df["feature"].isin(
        SELECTED_NUMERIC_FEATURES
    ),
    "numeric_median_imputer",
    "categorical_most_frequent_imputer",
)
candidate_missingness_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_candidate_missingness_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

# The five newly reconstructed Stage 04 features must be complete.
assert candidate_panel[
    list(STAGE04_BREADTH_FEATURES)
].notna().all().all()

# Legacy/base features may contain causal missing values, exactly as in
# Stage 06/07. No selected feature may be completely absent.
assert missing_by_feature.lt(
    len(candidate_panel)
).all()

assert not np.isinf(
    candidate_panel[
        SELECTED_NUMERIC_FEATURES
    ].to_numpy(dtype=float)
).any()

candidate_fingerprint = dataframe_fingerprint(
    candidate_panel,
    ["event_id", "symbol", "dEven", *SELECTED_FEATURES],
)
candidate_panel_audit_df = pd.DataFrame([{
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "unseen_test_rows": total_test_rows,
    "unseen_test_eligible_events": total_test_eligible,
    "candidate_events": int(len(candidate_panel)),
    "symbols_with_candidates": int(candidate_panel["symbol"].nunique()),
    "signal_dates": int(candidate_panel["dEven"].nunique()),
    "first_candidate_date": candidate_panel["dEven"].min(),
    "last_candidate_date": candidate_panel["dEven"].max(),
    "duplicate_event_ids": int(candidate_panel["event_id"].duplicated().sum()),
    "selected_feature_set": SELECTED_FEATURE_SET,
    "raw_model_features": len(SELECTED_FEATURES),
    "numeric_model_features": len(SELECTED_NUMERIC_FEATURES),
    "categorical_model_features": len(SELECTED_CATEGORICAL_FEATURES),
    "selected_feature_missing_values": int(
        missing_by_feature.sum()
    ),
    "features_with_missing_values": int(
        missing_by_feature.gt(0).sum()
    ),
    "stage04_added_feature_missing_values": int(
        candidate_panel[
            list(STAGE04_BREADTH_FEATURES)
        ].isna().sum().sum()
    ),
    "frozen_pipeline_imputation_used": True,
    "started_source_counts": json.dumps(started_source_counts, sort_keys=True),
    "candidate_population_sha256": candidate_fingerprint,
    "outcome_columns_loaded": False,
}])
candidate_panel_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)
display(candidate_panel_audit_df)


[test candidates] [   1/499] elapsed=0.6s errors=0
[test candidates] [  25/499] elapsed=10.0s errors=0
[test candidates] [  50/499] elapsed=20.7s errors=0
[test candidates] [  75/499] elapsed=31.3s errors=0
[test candidates] [ 100/499] elapsed=41.3s errors=0
[test candidates] [ 125/499] elapsed=51.8s errors=0
[test candidates] [ 150/499] elapsed=62.3s errors=0
[test candidates] [ 175/499] elapsed=72.2s errors=0
[test candidates] [ 200/499] elapsed=83.1s errors=0
[test candidates] [ 225/499] elapsed=92.8s errors=0
[test candidates] [ 250/499] elapsed=102.9s errors=0
[test candidates] [ 275/499] elapsed=112.6s errors=0
[test candidates] [ 300/499] elapsed=122.9s errors=0
[test candidates] [ 325/499] elapsed=132.9s errors=0
[test candidates] [ 350/499] elapsed=143.3s errors=0
[test candidates] [ 375/499] elapsed=153.8s errors=0
[test candidates] [ 400/499] elapsed=163.7s errors=0
[test candidates] [ 425/499] elapsed=174.0s errors=0
[test candidates] [ 450/499] elapsed=183.8s errors=0
[tes

,scientific_status,unseen_test_rows,unseen_test_eligible_events,candidate_events,symbols_with_candidates,signal_dates,first_candidate_date,last_candidate_date,duplicate_event_ids,selected_feature_set,raw_model_features,numeric_model_features,categorical_model_features,selected_feature_missing_values,features_with_missing_values,stage04_added_feature_missing_values,frozen_pipeline_imputation_used,started_source_counts,candidate_population_sha256,outcome_columns_loaded
0,post_hoc_retest_previously_inspected_period,371321,359454,78189,499,820,2021-03-27,2024-09-11,0,I_full_40,40,38,2,57,1,0,True,"{""raw_data"": 499}",33a2232d934fc9c17e02f437395ae6231666ade4e079bb...,False


## 6. Frozen inference and outcome-free abstention-policy lock


In [7]:
frozen_pipeline = joblib.load(model_path)
transformed_feature_names = [
    str(value)
    for value in frozen_pipeline.named_steps[
        "preprocess"
    ].get_feature_names_out()
]

assert len(transformed_feature_names) == int(
    stage09_config["frozen_inputs"]["expected_transformed_features"]
)
assert len(transformed_feature_names) == 47

inference_started = time.perf_counter()
raw_scores = frozen_pipeline.predict_proba(
    candidate_panel[SELECTED_FEATURES]
)[:, 1]
inference_seconds = time.perf_counter() - inference_started

assert np.isfinite(raw_scores).all()
assert ((raw_scores >= 0.0) & (raw_scores <= 1.0)).all()

inference_frame = candidate_panel[
    [
        "event_id",
        "symbol",
        "dEven",
        "market_breadth_regime",
    ]
].copy()
inference_frame["xgboost_ranking_score"] = raw_scores

inference_lock_df = apply_abstention_policy_inference(
    inference_frame,
    policy=FROZEN_ABSTENTION_POLICY,
)

# Structural comparison only: the old policy selected
# max(1, ceil(5% * same-day candidates)) on every date.
inference_lock_df[
    "old_top5_minimum_one_quota"
] = np.maximum(
    1,
    np.ceil(
        inference_lock_df[
            "daily_candidate_count"
        ].to_numpy(dtype=float)
        * POLICY_MAXIMUM_DAILY_FRACTION
    ).astype(int),
)
old_top5_baseline_signals = int(
    inference_lock_df.groupby(
        "dEven",
        observed=False,
    )[
        "old_top5_minimum_one_quota"
    ].first().sum()
)
selected_signals_locked = int(
    inference_lock_df["selected_signal"].sum()
)
dates_with_signal_locked = int(
    inference_lock_df.loc[
        inference_lock_df["selected_signal"],
        "dEven",
    ].nunique()
)
signal_dates_locked = int(
    inference_lock_df["dEven"].nunique()
)
zero_signal_dates_locked = int(
    signal_dates_locked - dates_with_signal_locked
)
signal_coverage_vs_old_top5 = float(
    selected_signals_locked
    / old_top5_baseline_signals
)

lock_columns = [
    "event_id",
    "symbol",
    "dEven",
    "market_breadth_regime",
    "xgboost_ranking_score",
    "daily_candidate_count",
    "daily_maximum_quota",
    "market_gate_pass",
    "score_threshold_pass",
    "policy_eligible",
    "daily_eligible_count",
    "daily_eligible_rank",
    "daily_signal_quota",
    "daily_selected_score_cutoff",
    "old_top5_minimum_one_quota",
    "selected_signal",
]
lock_path = (
    REPOSITORY_ROOT
    / stage09_config["inference_lock"]["lock_file"]
)
lock_path.parent.mkdir(parents=True, exist_ok=True)
inference_lock_df[lock_columns].to_csv(
    lock_path,
    index=False,
    encoding="utf-8-sig",
)
lock_file_hash = file_sha256(lock_path)

assert len(inference_lock_df) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert signal_dates_locked == int(
    stage09_config["frozen_inputs"]["expected_signal_dates"]
)
assert selected_signals_locked > 0
assert selected_signals_locked <= old_top5_baseline_signals
assert inference_lock_df["selected_signal"].sum() == (
    inference_lock_df.groupby("dEven")[
        "daily_signal_quota"
    ].first().sum()
)
assert inference_lock_df.groupby("dEven")[
    "selected_signal"
].sum().le(
    inference_lock_df.groupby("dEven")[
        "daily_maximum_quota"
    ].first()
).all()
assert not inference_lock_df.loc[
    ~inference_lock_df["policy_eligible"],
    "selected_signal",
].any()
assert inference_lock_df.loc[
    inference_lock_df["selected_signal"],
    "market_breadth_regime",
].isin(POLICY_ALLOWED_REGIMES).all()
assert inference_lock_df.loc[
    inference_lock_df["selected_signal"],
    "xgboost_ranking_score",
].ge(POLICY_MINIMUM_RAW_SCORE).all()

inference_lock_manifest = {
    "stage": 9,
    "status": "outcome_free_abstention_inference_locked",
    "scientific_status": (
        "post_hoc_retest_previously_inspected_period"
    ),
    "confirmatory_claim_allowed": False,
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "model_sha256": actual_model_hash,
    "selected_model": SELECTED_MODEL,
    "selected_feature_set": SELECTED_FEATURE_SET,
    "selected_trial": SELECTED_TRIAL,
    "raw_feature_count": len(SELECTED_FEATURES),
    "transformed_feature_count": len(transformed_feature_names),
    "candidate_population_sha256": candidate_fingerprint,
    "stage08_policy_id": STAGE08_POLICY_ID,
    "stage08_policy_file_sha256": STAGE08_POLICY_FILE_SHA256,
    "stage08_policy_configuration_hash": (
        STAGE08_POLICY_CONFIGURATION_HASH
    ),
    "inference_lock_file": str(lock_path),
    "inference_lock_file_sha256": lock_file_hash,
    "candidate_events": int(len(inference_lock_df)),
    "signal_dates": signal_dates_locked,
    "dates_with_signal": dates_with_signal_locked,
    "zero_signal_dates": zero_signal_dates_locked,
    "selected_signals": selected_signals_locked,
    "old_top5_minimum_one_signals": old_top5_baseline_signals,
    "signal_coverage_vs_old_top5": signal_coverage_vs_old_top5,
    "policy": {
        "policy_type": (
            "breadth_regime_gate_raw_score_threshold_"
            "daily_top_fraction_cap"
        ),
        "gate_name": POLICY_GATE_NAME,
        "allowed_regimes": list(POLICY_ALLOWED_REGIMES),
        "minimum_raw_score": POLICY_MINIMUM_RAW_SCORE,
        "threshold_source_quantile": (
            POLICY_THRESHOLD_SOURCE_QUANTILE
        ),
        "maximum_daily_fraction": (
            POLICY_MAXIMUM_DAILY_FRACTION
        ),
        "minimum_signals_per_date": (
            POLICY_MINIMUM_SIGNALS_PER_DATE
        ),
        "zero_signal_dates_allowed": True,
        "ranking_order": POLICY_RANKING_ORDER,
        "score_interpretation": (
            "uncalibrated_ranking_score"
        ),
        "calibration_method": "raw_identity",
        "fixed_raw_score_threshold_selected": True,
        "fixed_probability_threshold_selected": False,
    },
    "outcome_columns_in_lock_file": False,
    "test_labels_loaded_before_lock": False,
    "test_outcomes_loaded_before_lock": False,
    "policy_reselected_in_stage09": False,
    "model_retrained": False,
    "inference_seconds": float(inference_seconds),
}

lock_manifest_path = (
    REPOSITORY_ROOT
    / stage09_config["inference_lock"]["lock_manifest"]
)
lock_manifest_path.parent.mkdir(parents=True, exist_ok=True)
lock_manifest_path.write_text(
    json.dumps(
        inference_lock_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Outcome-free abstention lock:", lock_path)
print("Lock SHA256:", lock_file_hash)
print("Candidate events:", len(inference_lock_df))
print("Old Top-5%-minimum-one signals:", old_top5_baseline_signals)
print("Frozen abstention-policy signals:", selected_signals_locked)
print("Signal coverage vs old policy:", signal_coverage_vs_old_top5)
print("Zero-signal dates:", zero_signal_dates_locked)
print("Test labels loaded before lock: False")
print("Expected performance metrics supplied before lock: False")


Outcome-free abstention lock: E:\Iran_stock_trade\financial-ml-decision-support\results\predictions\09_abstention_inference_lock.csv
Lock SHA256: 9ae4d338601c7d364a8b2c2d36fd46e3bff746676ecb2a1e81c2b8490872977a
Candidate events: 78189
Old Top-5%-minimum-one signals: 4311
Frozen abstention-policy signals: 1815
Signal coverage vs old policy: 0.4210160055671538
Zero-signal dates: 445
Test labels loaded before lock: False
Expected performance metrics supplied before lock: False


## 7. Open outcomes and reconstruct corrected event returns after the lock

Signal dates remain capped at 22 September 2024. Raw observations through
26 October 2024 may only complete the already-defined 30-observation outcome
window. No post-cutoff row is used for features, scores, or signal selection.

In [8]:
outcome_columns = [
    "dEven", "eligible_for_modeling", "label", "event_end_date",
    "event_return", "holding_period_observations", "barrier_touched",
    "label_status", "same_bar_double_touch",
]
outcome_parts = []
outcome_error_rows = []
candidate_event_ids = set(inference_lock_df["event_id"].astype(str))
selected_event_ids = set(
    inference_lock_df.loc[
        inference_lock_df["selected_signal"].astype(bool), "event_id"
    ].astype(str)
)

for symbol in frozen_symbols:
    path = test_file_map[symbol]
    try:
        frame = pd.read_csv(path, usecols=outcome_columns, low_memory=False)
        frame["dEven"] = pd.to_datetime(
            frame["dEven"], errors="raise"
        ).dt.normalize()
        frame = frame.loc[
            frame["eligible_for_modeling"].fillna(False).astype(bool)
        ].copy()
        frame["event_id"] = symbol + "::" + frame["dEven"].dt.strftime("%Y-%m-%d")
        frame["symbol"] = symbol
        frame = frame.loc[frame["event_id"].isin(candidate_event_ids)].copy()
        outcome_parts.append(frame)
    except Exception as exc:
        outcome_error_rows.append({
            "symbol": symbol,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

outcome_errors_df = pd.DataFrame(
    outcome_error_rows,
    columns=["symbol", "error_type", "error_message"],
)
outcome_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_outcome_join_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not outcome_errors_df.empty:
    raise RuntimeError("Unseen-test outcome-loading errors exist.")

outcome_panel = pd.concat(outcome_parts, ignore_index=True)
outcome_panel["meta_label"] = pd.to_numeric(
    outcome_panel["label"], errors="raise"
).astype(int)
outcome_panel["event_end_date"] = pd.to_datetime(
    outcome_panel["event_end_date"], errors="raise"
).dt.normalize()
assert outcome_panel["event_id"].is_unique
assert set(outcome_panel["event_id"].astype(str)) == candidate_event_ids
assert outcome_panel["dEven"].le(SIGNAL_GENERATION_END).all()
assert outcome_panel["event_end_date"].le(SIGNAL_GENERATION_END).all()

# Corrected economic outcomes are reconstructed only after the frozen
# outcome-free score and signal lock exists. Predictive and signal-
# classification metrics use the complete unchanged candidate population.
selected_outcome_input = outcome_panel.loc[
    outcome_panel["event_id"].isin(selected_event_ids),
    [
        "event_id", "symbol", "dEven", "meta_label",
        "barrier_touched", "event_return", "event_end_date",
    ],
].copy()
assert len(selected_outcome_input) == selected_signals_locked
assert selected_outcome_input["event_id"].is_unique
assert set(selected_outcome_input["event_id"].astype(str)) == selected_event_ids

corrected_outcome_panel, corrected_outcome_errors_df = (
    reconstruct_corrected_event_outcomes(
        selected_outcome_input,
        raw_file_map=raw_file_map,
        signal_generation_end=SIGNAL_GENERATION_END,
        outcome_observation_tail_end=OUTCOME_OBSERVATION_TAIL_END,
        horizon_observations=int(
            stage09_config["corrected_event_outcome_policy"][
                "horizon_observations"
            ]
        ),
        upper_barrier_return=float(
            stage09_config["corrected_event_outcome_policy"][
                "upper_barrier_return"
            ]
        ),
        lower_barrier_return=float(
            stage09_config["corrected_event_outcome_policy"][
                "lower_barrier_return"
            ]
        ),
    )
)
corrected_outcome_errors_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_corrected_event_return_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
if not corrected_outcome_errors_df.empty:
    display(
        corrected_outcome_errors_df.groupby(
            ["error_type", "error_message"], dropna=False
        ).size().reset_index(name="events").head(20)
    )
    raise RuntimeError("Corrected selected-signal return reconstruction errors exist.")

assert corrected_outcome_panel["event_id"].is_unique
assert set(corrected_outcome_panel["event_id"].astype(str)) == selected_event_ids

evaluation_df = inference_lock_df[lock_columns].merge(
    outcome_panel[[
        "event_id", "meta_label", "event_end_date", "event_return",
        "holding_period_observations", "barrier_touched", "label_status",
        "same_bar_double_touch",
    ]],
    on="event_id",
    how="left",
    validate="one_to_one",
).merge(
    corrected_outcome_panel,
    on="event_id",
    how="left",
    validate="one_to_one",
)
evaluation_df["calendar_year"] = evaluation_df["dEven"].dt.year.astype(int)
evaluation_df = evaluation_df.rename(
    columns={"event_return": "original_event_return"}
)

assert evaluation_df["meta_label"].isin([0, 1]).all()
assert evaluation_df["label_status"].eq("labeled").all()
assert evaluation_df["original_event_return"].notna().all()
assert evaluation_df["event_end_date"].ge(evaluation_df["dEven"]).all()
assert evaluation_df["event_end_date"].le(SIGNAL_GENERATION_END).all()

selected_evaluation_df = evaluation_df.loc[
    evaluation_df["selected_signal"].astype(bool)
].copy()
assert len(selected_evaluation_df) == selected_signals_locked
assert selected_evaluation_df["corrected_event_return"].notna().all()
assert selected_evaluation_df["outcome_window_end_date"].le(
    OUTCOME_OBSERVATION_TAIL_END
).all()
assert (
    selected_evaluation_df["corrected_event_return"].gt(0.0).astype(int)
    == selected_evaluation_df["meta_label"].astype(int)
).all()

full_output_path = (
    REPOSITORY_ROOT
    / stage09_config["outputs"]["full_evaluation_file"]
)
selected_output_path = (
    REPOSITORY_ROOT
    / stage09_config["outputs"]["selected_signals_file"]
)
full_output_path.parent.mkdir(parents=True, exist_ok=True)
selected_output_path.parent.mkdir(parents=True, exist_ok=True)
evaluation_df.to_csv(full_output_path, index=False, encoding="utf-8-sig")
selected_evaluation_df.to_csv(
    selected_output_path,
    index=False,
    encoding="utf-8-sig",
)

selected_tail_mask = selected_evaluation_df[
    "outcome_window_uses_tail"
].astype(bool)
outcome_tail_audit_df = pd.DataFrame([{
    "signal_generation_start": TEST_START,
    "signal_generation_end": SIGNAL_GENERATION_END,
    "outcome_observation_tail_end": OUTCOME_OBSERVATION_TAIL_END,
    "candidate_events_for_predictive_evaluation": int(len(evaluation_df)),
    "corrected_outcome_reconstruction_population": (
        "frozen_abstention_policy_selected_signals_only"
    ),
    "corrected_outcome_events": int(len(selected_evaluation_df)),
    "selected_signals": int(evaluation_df["selected_signal"].sum()),
    "selected_signals_using_outcome_tail": int(selected_tail_mask.sum()),
    "latest_signal_date": evaluation_df["dEven"].max(),
    "latest_required_selected_outcome_window_end": (
        selected_evaluation_df["outcome_window_end_date"].max()
    ),
    "signals_generated_after_frozen_end": int(
        evaluation_df["dEven"].gt(SIGNAL_GENERATION_END).sum()
    ),
    "selected_outcome_windows_after_frozen_tail": int(
        selected_evaluation_df["outcome_window_end_date"]
        .gt(OUTCOME_OBSERVATION_TAIL_END).sum()
    ),
    "post_signal_end_rows_used_for_features_or_scoring": False,
}])
outcome_tail_audit_df.to_csv(
    RESULT_PATHS["audits"] / "09_abstention_outcome_observation_tail_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Candidate outcome labels joined:", len(evaluation_df))
print("Corrected selected-signal outcomes reconstructed:", len(selected_evaluation_df))
print("Selected signals using the outcome tail:", int(selected_tail_mask.sum()))
print(
    "Latest required selected outcome-window date:",
    selected_evaluation_df["outcome_window_end_date"].max().date(),
)


Candidate outcome labels joined: 78189
Corrected selected-signal outcomes reconstructed: 1815
Selected signals using the outcome tail: 27
Latest required selected outcome-window date: 2024-10-26


## 8. Post-hoc predictive, abstention-classification, and corrected outcome metrics

These metrics describe the exact frozen Stage 08 policy on the previously
inspected period. No Stage 09 metric is used to alter the gate, score cutoff,
daily cap, or model.


In [9]:
predictive_metric_values = probability_metrics(
    evaluation_df["meta_label"].to_numpy(dtype=int),
    evaluation_df["xgboost_ranking_score"].to_numpy(dtype=float),
    ece_bins=10,
)
predictive_metrics_df = pd.DataFrame([{
    **predictive_metric_values,
    "events": int(len(evaluation_df)),
    "score_interpretation": "uncalibrated_ranking_score",
    "literal_probability_interpretation_allowed": False,
    "calibration_fitted": False,
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_evaluation": False,
}])
predictive_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_predictive_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_classification = binary_signal_metrics(
    evaluation_df["meta_label"].to_numpy(dtype=int),
    evaluation_df["selected_signal"].to_numpy(dtype=bool),
)
signal_classification_metrics_df = pd.DataFrame([{
    **signal_classification,
    "policy_type": (
        "breadth_regime_gate_raw_score_threshold_"
        "daily_top_fraction_cap"
    ),
    "stage08_policy_id": STAGE08_POLICY_ID,
    "breadth_gate_name": POLICY_GATE_NAME,
    "allowed_regimes": json.dumps(
        list(POLICY_ALLOWED_REGIMES),
        ensure_ascii=False,
    ),
    "minimum_raw_score": POLICY_MINIMUM_RAW_SCORE,
    "threshold_source_quantile": POLICY_THRESHOLD_SOURCE_QUANTILE,
    "maximum_daily_fraction": POLICY_MAXIMUM_DAILY_FRACTION,
    "minimum_signals_per_date": POLICY_MINIMUM_SIGNALS_PER_DATE,
    "old_top5_minimum_one_signals": old_top5_baseline_signals,
    "selected_signals": selected_signals_locked,
    "signal_coverage_vs_old_top5": signal_coverage_vs_old_top5,
    "dates_with_signal": dates_with_signal_locked,
    "zero_signal_dates": zero_signal_dates_locked,
    "fixed_raw_score_threshold_selected": True,
    "fixed_probability_threshold_selected": False,
    "policy_changed_after_test_outcomes": False,
    "policy_reselected_in_stage09": False,
    "scientific_status": "post_hoc_retest_previously_inspected_period",
    "confirmatory_claim_allowed": False,
}])
signal_classification_metrics_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_classification_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_outcome_metrics = corrected_event_outcome_metrics(
    selected_evaluation_df
)
signal_outcome_summary_df = pd.DataFrame([{
    "population": "frozen_abstention_policy_selected_signals",
    **selected_outcome_metrics,
}])
signal_outcome_summary_df["return_interpretation"] = (
    "corrected_gross_event_level_outcome_not_portfolio_return"
)
signal_outcome_summary_df["upper_return_is_ex_post_maximum"] = True
signal_outcome_summary_df["transaction_costs_applied"] = False
signal_outcome_summary_df["portfolio_backtest_performed"] = False
signal_outcome_summary_df["scientific_status"] = (
    "post_hoc_retest_previously_inspected_period"
)
signal_outcome_summary_df["confirmatory_claim_allowed"] = False
signal_outcome_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_outcome_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_outcomes_by_year_df = grouped_corrected_event_outcome_metrics(
    selected_evaluation_df,
    group_column="calendar_year",
)
signal_outcomes_by_year_df[
    "maximum_daily_fraction"
] = POLICY_MAXIMUM_DAILY_FRACTION
signal_outcomes_by_year_df["portfolio_backtest_performed"] = False
signal_outcomes_by_year_df["confirmatory_claim_allowed"] = False
signal_outcomes_by_year_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_outcomes_by_year.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_outcomes_by_barrier_df = grouped_corrected_event_outcome_metrics(
    selected_evaluation_df,
    group_column="barrier_touched",
)
signal_outcomes_by_barrier_df["portfolio_backtest_performed"] = False
signal_outcomes_by_barrier_df["confirmatory_claim_allowed"] = False
signal_outcomes_by_barrier_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_outcomes_by_barrier.csv",
    index=False,
    encoding="utf-8-sig",
)

signal_outcomes_by_regime_df = grouped_corrected_event_outcome_metrics(
    selected_evaluation_df,
    group_column="market_breadth_regime",
)
signal_outcomes_by_regime_df["portfolio_backtest_performed"] = False
signal_outcomes_by_regime_df["confirmatory_claim_allowed"] = False
signal_outcomes_by_regime_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_outcomes_by_regime.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_outcome_row = signal_outcome_summary_df.iloc[0]

# No expected performance value is asserted after the outcome-free lock.
for metric_name in [
    "win_rate",
    "average_winning_return",
    "average_losing_return",
    "payoff_ratio",
    "profit_factor",
    "mean_corrected_event_return",
    "median_corrected_event_return",
]:
    assert np.isfinite(float(selected_outcome_row[metric_name])), metric_name

assert int(selected_outcome_row["events"]) == selected_signals_locked
assert int(
    selected_outcome_row["winning_events"]
    + selected_outcome_row["losing_events_negative_return"]
    + selected_outcome_row["breakeven_events"]
) == int(selected_outcome_row["events"])

display(predictive_metrics_df)
display(signal_classification_metrics_df)
display(signal_outcome_summary_df)
display(signal_outcomes_by_year_df)
display(signal_outcomes_by_barrier_df)
display(signal_outcomes_by_regime_df)


,roc_auc,average_precision,brier_score,log_loss,expected_calibration_error,mean_predicted_score,observed_positive_fraction,events,score_interpretation,literal_probability_interpretation_allowed,calibration_fitted,scientific_status,confirmatory_evaluation
0,0.555947,0.500211,0.250232,0.694137,0.068775,0.514784,0.446009,78189,uncalibrated_ranking_score,False,False,post_hoc_retest_previously_inspected_period,False


,events,signals,signal_rate,true_positive,false_positive,true_negative,false_negative,precision,prevalence,precision_lift,...,selected_signals,signal_coverage_vs_old_top5,dates_with_signal,zero_signal_dates,fixed_raw_score_threshold_selected,fixed_probability_threshold_selected,policy_changed_after_test_outcomes,policy_reselected_in_stage09,scientific_status,confirmatory_claim_allowed
0,78189,1815,0.023213,1023,792,42524,33850,0.563636,0.446009,0.117627,...,1815,0.421016,375,445,True,False,False,False,post_hoc_retest_previously_inspected_period,False


,population,events,winning_events,nonwinning_events,losing_events_negative_return,breakeven_events,win_rate,mean_corrected_event_return,median_corrected_event_return,corrected_event_return_standard_deviation,...,lower_barrier_events,vertical_barrier_events,mean_label_event_holding_period_observations,median_label_event_holding_period_observations,return_interpretation,upper_return_is_ex_post_maximum,transaction_costs_applied,portfolio_backtest_performed,scientific_status,confirmatory_claim_allowed
0,frozen_abstention_policy_selected_signals,1815,1023,792,791,1,0.563636,0.100765,0.039091,0.244953,...,409,635,19.740496,20.0,corrected_gross_event_level_outcome_not_portfo...,True,False,False,post_hoc_retest_previously_inspected_period,False


,calendar_year,events,winning_events,nonwinning_events,losing_events_negative_return,breakeven_events,win_rate,mean_corrected_event_return,median_corrected_event_return,corrected_event_return_standard_deviation,...,gross_loss_absolute_sum,profit_factor,upper_barrier_events,lower_barrier_events,vertical_barrier_events,mean_label_event_holding_period_observations,median_label_event_holding_period_observations,maximum_daily_fraction,portfolio_backtest_performed,confirmatory_claim_allowed
0,2021,298,140,158,158,0,0.469799,0.060698,-0.021555,0.224057,...,19.370986,1.933769,120,108,70,16.758389,15.0,0.05,False,False
1,2022,677,488,189,189,0,0.720827,0.182712,0.182735,0.247459,...,19.082781,7.482068,396,93,188,18.946824,19.0,0.05,False,False
2,2023,483,268,215,214,1,0.554865,0.086347,0.019774,0.264886,...,22.302339,2.870008,166,111,206,20.875776,24.0,0.05,False,False
3,2024,357,127,230,230,0,0.355742,-0.001685,-0.053592,0.165044,...,23.585526,0.974497,89,97,171,22.198880,29.0,0.05,False,False


,barrier_touched,events,winning_events,nonwinning_events,losing_events_negative_return,breakeven_events,win_rate,mean_corrected_event_return,median_corrected_event_return,corrected_event_return_standard_deviation,...,gross_profit_sum,gross_loss_absolute_sum,profit_factor,upper_barrier_events,lower_barrier_events,vertical_barrier_events,mean_label_event_holding_period_observations,median_label_event_holding_period_observations,portfolio_backtest_performed,confirmatory_claim_allowed
0,lower,409,0,409,409,0,0.00000,-0.150000,-0.150000,2.775558e-17,...,0.000000,61.350000,0.000000,0,409,0,15.413203,15.0,False,False
1,upper,771,771,0,0,0,1.00000,0.329881,0.263493,2.015911e-01,...,254.338251,-0.000000,NaN,771,0,0,13.586252,13.0,False,False
2,vertical,635,252,383,382,1,0.39685,-0.015906,-0.020229,6.562454e-02,...,12.891323,22.991631,0.560696,0,0,635,30.000000,30.0,False,False


,market_breadth_regime,events,winning_events,nonwinning_events,losing_events_negative_return,breakeven_events,win_rate,mean_corrected_event_return,median_corrected_event_return,corrected_event_return_standard_deviation,...,gross_profit_sum,gross_loss_absolute_sum,profit_factor,upper_barrier_events,lower_barrier_events,vertical_barrier_events,mean_label_event_holding_period_observations,median_label_event_holding_period_observations,portfolio_backtest_performed,confirmatory_claim_allowed
0,recovery_negative,1337,711,626,626,0,0.531788,0.073408,0.018349,0.219947,...,164.545130,66.398584,2.478142,505,319,513,20.240090,22.0,False,False
1,recovery_positive,478,312,166,165,1,0.652720,0.177283,0.172945,0.290808,...,102.684444,17.943047,5.722799,266,90,122,18.343096,18.0,False,False


## 9. Daily abstention and Breadth-regime implementation audit


In [10]:
signal_date_audit_df = evaluation_df.groupby(
    "dEven",
    as_index=False,
).agg(
    market_breadth_regime=("market_breadth_regime", "first"),
    candidate_events=("event_id", "size"),
    gate_pass_events=("market_gate_pass", "sum"),
    threshold_pass_events=("score_threshold_pass", "sum"),
    policy_eligible_events=("policy_eligible", "sum"),
    daily_maximum_quota=("daily_maximum_quota", "first"),
    required_signal_quota=("daily_signal_quota", "first"),
    selected_signals=("selected_signal", "sum"),
    score_cutoff=("daily_selected_score_cutoff", "first"),
    minimum_score=("xgboost_ranking_score", "min"),
    maximum_score=("xgboost_ranking_score", "max"),
    selected_positive_labels=(
        "meta_label",
        lambda series: int(
            series[
                evaluation_df.loc[
                    series.index,
                    "selected_signal",
                ].to_numpy(dtype=bool)
            ].sum()
        ),
    ),
)
daily_selected_return_df = (
    selected_evaluation_df.groupby("dEven", as_index=False)
    .agg(
        selected_mean_corrected_event_return=(
            "corrected_event_return",
            "mean",
        )
    )
)
signal_date_audit_df = signal_date_audit_df.merge(
    daily_selected_return_df,
    on="dEven",
    how="left",
    validate="one_to_one",
)
signal_date_audit_df["zero_signal_date"] = (
    signal_date_audit_df["selected_signals"].eq(0)
)
signal_date_audit_df["selected_precision"] = np.where(
    signal_date_audit_df["selected_signals"].gt(0),
    (
        signal_date_audit_df["selected_positive_labels"]
        / signal_date_audit_df["selected_signals"]
    ),
    np.nan,
)

assert signal_date_audit_df["selected_signals"].eq(
    signal_date_audit_df["required_signal_quota"]
).all()
assert signal_date_audit_df["selected_signals"].le(
    signal_date_audit_df["daily_maximum_quota"]
).all()
assert int(
    signal_date_audit_df["selected_signals"].sum()
) == selected_signals_locked
assert int(
    signal_date_audit_df["zero_signal_date"].sum()
) == zero_signal_dates_locked
assert signal_date_audit_df.loc[
    signal_date_audit_df["selected_signals"].gt(0),
    "selected_mean_corrected_event_return",
].notna().all()
assert signal_date_audit_df.loc[
    signal_date_audit_df["zero_signal_date"],
    "selected_mean_corrected_event_return",
].isna().all()

signal_date_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_signal_date_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

regime_selection_rows = []
for regime, group in evaluation_df.groupby(
    "market_breadth_regime",
    sort=True,
    observed=False,
):
    selected_mask = group["selected_signal"].astype(bool)
    selected_labels = group.loc[
        selected_mask,
        "meta_label",
    ]
    regime_selection_rows.append({
        "market_breadth_regime": str(regime),
        "candidate_events": int(len(group)),
        "signal_dates": int(group["dEven"].nunique()),
        "selected_signals": int(selected_mask.sum()),
        "selected_positive_labels": int(selected_labels.sum()),
        "selected_precision": (
            float(selected_labels.mean())
            if len(selected_labels) > 0
            else np.nan
        ),
        "allowed_by_frozen_gate": bool(
            str(regime) in POLICY_ALLOWED_REGIMES
        ),
    })
regime_selection_audit_df = pd.DataFrame(
    regime_selection_rows
)
regime_selection_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "09_abstention_policy_regime_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert not regime_selection_audit_df.loc[
    ~regime_selection_audit_df["allowed_by_frozen_gate"],
    "selected_signals",
].gt(0).any()

display(signal_date_audit_df.head())
display(regime_selection_audit_df)


,dEven,market_breadth_regime,candidate_events,gate_pass_events,threshold_pass_events,policy_eligible_events,daily_maximum_quota,required_signal_quota,selected_signals,score_cutoff,minimum_score,maximum_score,selected_positive_labels,selected_mean_corrected_event_return,zero_signal_date,selected_precision
0,2021-03-27,broad_decline,89,0,89,0,5,0,0,NaN,0.425717,0.547557,0,NaN,True,NaN
1,2021-03-28,broad_decline,96,0,96,0,5,0,0,NaN,0.417818,0.583492,0,NaN,True,NaN
2,2021-03-30,broad_decline,90,0,90,0,5,0,0,NaN,0.395325,0.577911,0,NaN,True,NaN
3,2021-03-31,broad_decline,95,0,95,0,5,0,0,NaN,0.386635,0.585614,0,NaN,True,NaN
4,2021-04-03,broad_decline,94,0,94,0,5,0,0,NaN,0.391695,0.530262,0,NaN,True,NaN


,market_breadth_regime,candidate_events,signal_dates,selected_signals,selected_positive_labels,selected_precision,allowed_by_frozen_gate
0,broad_advance,39,13,0,0,NaN,False
1,broad_decline,6395,67,0,0,NaN,False
2,deterioration,39165,365,0,0,NaN,False
3,recovery_negative,24569,228,1337,711,0.531788,True
4,recovery_positive,8021,147,478,312,0.652720,True


## 10. Write the Stage 09 v5 abstention-policy post-hoc manifest


In [11]:
selected_outcome_row = signal_outcome_summary_df.iloc[0]

manifest = {
    "stage": 9,
    "status": "completed_internal_integrity_checks",
    "stage_version": (
        "v5_frozen_abstention_policy_post_hoc_retest_"
        "selected_signal_corrected_outcomes"
    ),
    "notebook": "09_signal_level_evaluation.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "scientific_status": {
        "classification": "post_hoc_retest_previously_inspected_period",
        "prior_test_period_previously_inspected": True,
        "breadth_design_informed_by_prior_test_diagnostics": True,
        "abstention_policy_selected_only_from_train_oof": True,
        "confirmatory_claim_allowed": False,
        "external_validation_claim_allowed": False,
        "future_confirmation_required": True,
        "recommended_confirmation": (
            "later data not used in design or prospective paper trading"
        ),
    },
    "lineage": {
        "stage04_candidate_threshold_fraction": PRIMARY_THRESHOLD,
        "stage07_code_commit_sha": stage07_manifest["git_commit_sha"],
        "stage07_model_sha256": actual_model_hash,
        "stage07_selected_model": SELECTED_MODEL,
        "stage07_selected_feature_set": SELECTED_FEATURE_SET,
        "stage07_selected_trial": SELECTED_TRIAL,
        "stage07_selected_raw_features": SELECTED_FEATURES,
        "stage08_code_commit_sha": stage08_manifest["git_commit_sha"],
        "stage08_policy_id": STAGE08_POLICY_ID,
        "stage08_policy_file_sha256": STAGE08_POLICY_FILE_SHA256,
        "stage08_policy_configuration_hash": (
            STAGE08_POLICY_CONFIGURATION_HASH
        ),
    },
    "temporal_scope": {
        "evaluation_period_start": str(TEST_START.date()),
        "signal_generation_end": str(SIGNAL_GENERATION_END.date()),
        "outcome_observation_tail_end": str(
            OUTCOME_OBSERVATION_TAIL_END.date()
        ),
        "signals_generated_in_outcome_tail": 0,
    },
    "candidate_population": {
        "events": int(len(evaluation_df)),
        "symbols": int(evaluation_df["symbol"].nunique()),
        "signal_dates": int(evaluation_df["dEven"].nunique()),
        "candidate_population_sha256": candidate_fingerprint,
        "selected_feature_set": SELECTED_FEATURE_SET,
        "raw_feature_count": len(SELECTED_FEATURES),
        "numeric_feature_count": len(SELECTED_NUMERIC_FEATURES),
        "categorical_feature_count": len(SELECTED_CATEGORICAL_FEATURES),
    },
    "inference_lock": inference_lock_manifest,
    "abstention_diagnostics": {
        "old_top5_minimum_one_signals": old_top5_baseline_signals,
        "selected_signals": selected_signals_locked,
        "signal_coverage_vs_old_top5": signal_coverage_vs_old_top5,
        "signal_dates": signal_dates_locked,
        "dates_with_signal": dates_with_signal_locked,
        "zero_signal_dates": zero_signal_dates_locked,
    },
    "corrected_event_outcome_policy": (
        stage09_config["corrected_event_outcome_policy"]
    ),
    "corrected_outcome_reconstruction_population": (
        "frozen_abstention_policy_selected_signals_only"
    ),
    "outcome_tail_audit": outcome_tail_audit_df.iloc[0].to_dict(),
    "predictive_metrics": predictive_metrics_df.iloc[0].to_dict(),
    "signal_classification_metrics": (
        signal_classification_metrics_df.iloc[0].to_dict()
    ),
    "selected_signal_corrected_outcomes": selected_outcome_row.to_dict(),
    "policy": {
        "source_stage": 8,
        "stage08_policy_id": STAGE08_POLICY_ID,
        "stage08_policy_configuration_hash": (
            STAGE08_POLICY_CONFIGURATION_HASH
        ),
        "policy_type": (
            "breadth_regime_gate_raw_score_threshold_"
            "daily_top_fraction_cap"
        ),
        "gate_name": POLICY_GATE_NAME,
        "allowed_regimes": list(POLICY_ALLOWED_REGIMES),
        "minimum_raw_score": POLICY_MINIMUM_RAW_SCORE,
        "threshold_source_quantile": (
            POLICY_THRESHOLD_SOURCE_QUANTILE
        ),
        "maximum_daily_fraction": (
            POLICY_MAXIMUM_DAILY_FRACTION
        ),
        "minimum_signals_per_date": (
            POLICY_MINIMUM_SIGNALS_PER_DATE
        ),
        "zero_signal_dates_allowed": True,
        "ranking_order": POLICY_RANKING_ORDER,
        "score_interpretation": "uncalibrated_ranking_score",
        "calibration_method": "raw_identity",
        "calibrator_fitted": False,
        "fixed_raw_score_threshold_selected": True,
        "fixed_probability_threshold_selected": False,
        "policy_reselected_in_stage09": False,
    },
    "safeguards": {
        "scores_and_signals_locked_before_outcome_join": True,
        "expected_performance_metrics_supplied_before_lock": False,
        "expected_selected_signal_count_supplied_before_lock": False,
        "expected_inference_lock_hash_supplied_before_run": False,
        "test_labels_used_before_inference_lock": False,
        "post_signal_end_rows_used_for_features_or_scoring": False,
        "outcome_tail_used_only_for_existing_signal_windows": True,
        "model_retrained": False,
        "hyperparameters_changed": False,
        "selected_feature_set_changed": False,
        "candidate_threshold_changed": False,
        "calibration_refit": False,
        "stage08_policy_changed_in_stage09": False,
        "policy_reselected_in_stage09": False,
        "candidate_generation_hard_started_filter_applied": False,
        "candidate_generation_hard_breadth_filter_applied": False,
        "post_model_breadth_regime_gate_applied": True,
        "fixed_raw_score_threshold_applied": True,
        "minimum_signals_per_date_is_zero": True,
        "portfolio_backtest_performed": False,
        "transaction_costs_applied": False,
        "executable_exit_rule_claim_allowed": False,
    },
    "configuration_hash": stable_object_hash({
        "stage09_config": stage09_config,
        "stage04_policy": stage04_policy,
        "stage07_model_sha256": actual_model_hash,
        "stage08_policy_file_sha256": STAGE08_POLICY_FILE_SHA256,
        "stage08_policy_configuration_hash": (
            STAGE08_POLICY_CONFIGURATION_HASH
        ),
        "candidate_population_sha256": candidate_fingerprint,
        "inference_lock_file_sha256": lock_file_hash,
        "selected_signal_corrected_outcomes": (
            selected_outcome_row.to_dict()
        ),
    }),
}

manifest_path = (
    REPOSITORY_ROOT
    / stage09_config["outputs"]["evaluation_manifest"]
)
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Stage 09 v5 manifest:", manifest_path)
print("Manifest code SHA:", manifest["git_commit_sha"])


Stage 09 v5 manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\09_abstention_signal_evaluation_manifest.json
Manifest code SHA: 9d18ab35786ecb844b68a51a2ac3b2b0566dd044


## 11. Final integrity checks and truthful interpretation


In [12]:
assert candidate_errors_df.empty
assert outcome_errors_df.empty
assert corrected_outcome_errors_df.empty
assert market_index_errors_df.empty

assert total_test_rows == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_rows"]
)
assert total_test_eligible == int(
    stage09_config["frozen_inputs"]["expected_unseen_test_eligible_events"]
)
assert len(evaluation_df) == int(
    stage09_config["frozen_inputs"]["expected_candidate_events"]
)
assert evaluation_df["event_id"].is_unique
assert evaluation_df["dEven"].between(
    TEST_START,
    SIGNAL_GENERATION_END,
).all()
assert evaluation_df["event_end_date"].le(
    SIGNAL_GENERATION_END
).all()
assert evaluation_df["meta_label"].isin([0, 1]).all()
assert np.isfinite(
    evaluation_df["xgboost_ranking_score"].to_numpy(dtype=float)
).all()

assert len(selected_evaluation_df) == selected_signals_locked
assert selected_evaluation_df["event_id"].is_unique
assert selected_evaluation_df[
    "outcome_window_end_date"
].le(OUTCOME_OBSERVATION_TAIL_END).all()
assert np.isfinite(
    selected_evaluation_df[
        "corrected_event_return"
    ].to_numpy(dtype=float)
).all()
assert (
    selected_evaluation_df[
        "corrected_event_return"
    ].gt(0.0).astype(int)
    == selected_evaluation_df["meta_label"].astype(int)
).all()

assert len(BASE_MODEL_FEATURES) == 35
assert len(SELECTED_FEATURES) == 40
assert len(SELECTED_NUMERIC_FEATURES) == 38
assert SELECTED_CATEGORICAL_FEATURES == [
    "gmma_state",
    "market_breadth_regime",
]
assert len(transformed_feature_names) == 47
assert actual_model_hash == stage07_manifest["model"]["model_sha256"]
assert actual_model_hash == stage09_config[
    "frozen_inputs"
]["expected_model_sha256"]

assert STAGE08_POLICY_ID == (
    stage09_config["frozen_inputs"][
        "expected_stage08_policy_id"
    ]
)
assert STAGE08_POLICY_CONFIGURATION_HASH == stage08_manifest[
    "selected_policy_configuration_hash"
]
assert np.isclose(
    POLICY_MAXIMUM_DAILY_FRACTION,
    0.05,
)
assert POLICY_MINIMUM_SIGNALS_PER_DATE == 0
assert POLICY_GATE_NAME == "G3_recovery_only"
assert list(POLICY_ALLOWED_REGIMES) == [
    "recovery_negative",
    "recovery_positive",
]
assert np.isclose(
    POLICY_MINIMUM_RAW_SCORE,
    0.3844631016254425,
)
assert np.isclose(
    POLICY_THRESHOLD_SOURCE_QUANTILE,
    0.0,
)

assert int(evaluation_df["selected_signal"].sum()) == (
    selected_signals_locked
)
assert int(
    evaluation_df.loc[
        evaluation_df["selected_signal"],
        "dEven",
    ].nunique()
) == dates_with_signal_locked
assert int(
    signal_date_audit_df["zero_signal_date"].sum()
) == zero_signal_dates_locked
assert not evaluation_df.loc[
    ~evaluation_df["policy_eligible"],
    "selected_signal",
].any()
assert evaluation_df.loc[
    evaluation_df["selected_signal"],
    "market_breadth_regime",
].isin(POLICY_ALLOWED_REGIMES).all()
assert evaluation_df.loc[
    evaluation_df["selected_signal"],
    "xgboost_ranking_score",
].ge(POLICY_MINIMUM_RAW_SCORE).all()

assert inference_lock_manifest[
    "outcome_columns_in_lock_file"
] is False
assert inference_lock_manifest[
    "test_labels_loaded_before_lock"
] is False
assert inference_lock_manifest[
    "test_outcomes_loaded_before_lock"
] is False
assert inference_lock_manifest[
    "policy_reselected_in_stage09"
] is False

assert manifest["scientific_status"][
    "confirmatory_claim_allowed"
] is False
assert manifest["scientific_status"][
    "future_confirmation_required"
] is True
assert manifest["safeguards"][
    "scores_and_signals_locked_before_outcome_join"
] is True
assert manifest["safeguards"][
    "expected_performance_metrics_supplied_before_lock"
] is False
assert manifest["safeguards"][
    "expected_selected_signal_count_supplied_before_lock"
] is False
assert manifest["safeguards"][
    "expected_inference_lock_hash_supplied_before_run"
] is False
assert manifest["safeguards"]["model_retrained"] is False
assert manifest["safeguards"]["hyperparameters_changed"] is False
assert manifest["safeguards"][
    "selected_feature_set_changed"
] is False
assert manifest["safeguards"][
    "candidate_threshold_changed"
] is False
assert manifest["safeguards"]["calibration_refit"] is False
assert manifest["safeguards"][
    "stage08_policy_changed_in_stage09"
] is False
assert manifest["safeguards"][
    "policy_reselected_in_stage09"
] is False
assert manifest["safeguards"][
    "candidate_generation_hard_started_filter_applied"
] is False
assert manifest["safeguards"][
    "candidate_generation_hard_breadth_filter_applied"
] is False
assert manifest["safeguards"][
    "post_model_breadth_regime_gate_applied"
] is True
assert manifest["safeguards"][
    "fixed_raw_score_threshold_applied"
] is True
assert manifest["safeguards"][
    "minimum_signals_per_date_is_zero"
] is True
assert manifest["safeguards"][
    "portfolio_backtest_performed"
] is False
assert manifest["safeguards"][
    "transaction_costs_applied"
] is False
assert manifest["safeguards"][
    "post_signal_end_rows_used_for_features_or_scoring"
] is False

assert signal_date_audit_df["selected_signals"].eq(
    signal_date_audit_df["required_signal_quota"]
).all()
assert signal_date_audit_df["selected_signals"].le(
    signal_date_audit_df["daily_maximum_quota"]
).all()

print("Notebook 09 v5 integrity checks: PASSED")
print(
    "Scientific status:",
    "post-hoc retest on previously inspected period",
)
print("Confirmatory claim allowed: False")
print("Frozen model SHA256:", actual_model_hash)
print("Stage 08 policy ID:", STAGE08_POLICY_ID)
print("Stage 08 policy SHA256:", STAGE08_POLICY_FILE_SHA256)
print("Inference lock SHA256:", lock_file_hash)
print("Selected model:", SELECTED_MODEL)
print("Selected feature set:", SELECTED_FEATURE_SET)
print("Selected trial:", SELECTED_TRIAL)
print("Raw model features:", len(SELECTED_FEATURES))
print("Transformed model features:", len(transformed_feature_names))
print("Breadth gate:", POLICY_GATE_NAME)
print("Allowed regimes:", list(POLICY_ALLOWED_REGIMES))
print("Minimum raw score:", POLICY_MINIMUM_RAW_SCORE)
print(
    "Threshold source quantile:",
    POLICY_THRESHOLD_SOURCE_QUANTILE,
)
print(
    "Daily maximum fraction:",
    POLICY_MAXIMUM_DAILY_FRACTION,
)
print("Minimum signals per date:", POLICY_MINIMUM_SIGNALS_PER_DATE)
print("Evaluation-period raw rows:", total_test_rows)
print("Evaluation-period eligible events:", total_test_eligible)
print("Long candidate events:", len(evaluation_df))
print("Signal dates:", signal_dates_locked)
print("Old Top-5%-minimum-one signals:", old_top5_baseline_signals)
print("Frozen abstention-policy signals:", selected_signals_locked)
print("Signal coverage vs old policy:", signal_coverage_vs_old_top5)
print("Dates with signal:", dates_with_signal_locked)
print("Zero-signal dates:", zero_signal_dates_locked)
print(
    "Signal precision:",
    float(
        signal_classification_metrics_df.iloc[0]["precision"]
    ),
)
print(
    "Signal specificity:",
    float(
        signal_classification_metrics_df.iloc[0]["specificity"]
    ),
)
print(
    "Signal sensitivity:",
    float(
        signal_classification_metrics_df.iloc[0]["sensitivity"]
    ),
)
print(
    "Corrected selected-signal win rate:",
    float(selected_outcome_row["win_rate"]),
)
print(
    "Corrected selected-signal payoff ratio:",
    float(selected_outcome_row["payoff_ratio"]),
)
print(
    "Corrected selected-signal profit factor:",
    float(selected_outcome_row["profit_factor"]),
)
print(
    "Corrected selected-signal mean event return:",
    float(selected_outcome_row["mean_corrected_event_return"]),
)
print("Scores and policy decisions locked before outcomes: True")
print("Expected selected-signal count supplied before lock: False")
print("Expected performance values supplied before lock: False")
print("Policy reselected in Stage 09: False")
print("Portfolio backtest performed: False")
print("Transaction costs applied: False")
print(
    "Corrected return interpretation: gross selected-signal event outcome; "
    "upper outcomes use an ex-post 30-observation maximum and are not "
    "executable portfolio returns."
)
print(
    "Next action: independently audit this post-hoc abstention retest, "
    "then run an executable portfolio backtest without changing the "
    "frozen policy."
)


Notebook 09 v5 integrity checks: PASSED
Scientific status: post-hoc retest on previously inspected period
Confirmatory claim allowed: False
Frozen model SHA256: 73a781551cdabd6bb67f5b9c0836a8683372e46487d9aade29e6320006086c1f
Stage 08 policy ID: G3_recovery_only__q0000
Stage 08 policy SHA256: 549ef132549e56f693651a7350bb51c3dcb4e316769cb0165f30a65130ec127f
Inference lock SHA256: 9ae4d338601c7d364a8b2c2d36fd46e3bff746676ecb2a1e81c2b8490872977a
Selected model: xgboost
Selected feature set: I_full_40
Selected trial: 10
Raw model features: 40
Transformed model features: 47
Breadth gate: G3_recovery_only
Allowed regimes: ['recovery_negative', 'recovery_positive']
Minimum raw score: 0.3844631016254425
Threshold source quantile: 0.0
Daily maximum fraction: 0.05
Minimum signals per date: 0
Evaluation-period raw rows: 371321
Evaluation-period eligible events: 359454
Long candidate events: 78189
Signal dates: 820
Old Top-5%-minimum-one signals: 4311
Frozen abstention-policy signals: 1815
Signal 

## Review before freezing Stage 09 v5

Review all `09_abstention_` audits, the outcome-free lock, and the final
manifest. Do not alter the Stage 08 gate, raw-score cutoff, or daily cap after
viewing these results. The 2021–2024 period remains post-hoc; later unused data
or prospective paper trading is required for genuine confirmation.
